# 🏡 ナナカファーム 農作業小屋 3Dモデルビューア

**AI × Blender MCP で作成した農作業小屋を Open3D で表示します。**

| | 単管パイプ版（本番） | 単管パイプ版（約3畳） | 木造版（本番） | 木造版（約3畳） |
|---|---|---|---|---|
| 幅×奥行 | 6000×5000mm | 1820×2730mm | 6000×5000mm | 1820×2730mm |
| 南側軒高 | 3,000mm | 2,100mm | 3,000mm | 2,100mm |
| 北側軒高 | 2,500mm | 1,800mm | 2,500mm | 1,800mm |

📝 関連記事: [AI × Blender MCP で農作業小屋の設計図を1日で4種類作った話](https://zenn.dev/hiroakikody/articles/47ba166f4dad26)

## ⚡ クイックスタート
1. **Runtime → Run all** でセル全実行
2. Colab では静止画レンダリング（PNG）で出力
3. **ローカル実行**なら `draw_shed_interactive()` でぐりぐり操作できます

### ローカルでのインタラクティブ操作
```bash
pip install open3d
jupyter notebook view_models_open3d.ipynb
```
操作: 左ドラッグ=回転 / 右ドラッグ=移動 / スクロール=ズーム

In [ ]:
# ============================================================
# CELL 1: インストール
# ============================================================
!pip install open3d -q

import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

print(f'✅ Open3D version: {o3d.__version__}')
IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_BACKEND_VERSION' in os.environ
print(f'   実行環境: {"Google Colab" if IS_COLAB else "ローカル"}')

In [ ]:
# ============================================================
# CELL 2: ジオメトリ生成ユーティリティ
# ============================================================
# 単位はすべて mm → Open3D では m に変換（÷1000）

S = 0.001  # mm → m スケール係数

def box(cx, cy, cz, sx, sy, sz, color):
    """直方体メッシュ（中心座標 + 各辺長さ）"""
    m = o3d.geometry.TriangleMesh.create_box(sx*S, sy*S, sz*S)
    m.translate([(cx - sx/2)*S, (cy - sy/2)*S, (cz - sz/2)*S])
    m.paint_uniform_color(color)
    m.compute_vertex_normals()
    return m

def cylinder(cx, cy, cz, r, h, color, axis='z'):
    """円柱（単管パイプ用）"""
    m = o3d.geometry.TriangleMesh.create_cylinder(r*S, h*S, resolution=12)
    if axis == 'x':
        R = m.get_rotation_matrix_from_xyz([0, np.pi/2, 0])
        m.rotate(R, center=(0,0,0))
    elif axis == 'y':
        R = m.get_rotation_matrix_from_xyz([np.pi/2, 0, 0])
        m.rotate(R, center=(0,0,0))
    m.translate([cx*S, cy*S, cz*S])
    m.paint_uniform_color(color)
    m.compute_vertex_normals()
    return m

def pipe_between(p1, p2, r, color):
    """2点間を結ぶ単管パイプ（任意方向対応）"""
    p1, p2 = np.array(p1, float), np.array(p2, float)
    center = (p1 + p2) / 2
    diff   = p2 - p1
    length = np.linalg.norm(diff)
    m = o3d.geometry.TriangleMesh.create_cylinder(r*S, length*S, resolution=12)
    # Z軸 → diff 方向へ回転
    z = np.array([0, 0, 1.0])
    d = diff / length
    if not np.allclose(z, d) and not np.allclose(z, -d):
        axis  = np.cross(z, d)
        axis /= np.linalg.norm(axis)
        angle = np.arccos(np.clip(np.dot(z, d), -1, 1))
        R = m.get_rotation_matrix_from_axis_angle(axis * angle)
        m.rotate(R, center=(0,0,0))
    elif np.allclose(z, -d):  # 真逆方向
        m.rotate(m.get_rotation_matrix_from_xyz([np.pi, 0, 0]), center=(0,0,0))
    m.translate(center * S)
    m.paint_uniform_color(color)
    m.compute_vertex_normals()
    return m

def inclined_box(cx, cy, cz, sx, sy, sz, slope_rad, color):
    """傾斜した板（ポリカ屋根・野地板用）X軸周りに回転"""
    m = o3d.geometry.TriangleMesh.create_box(sx*S, sy*S, sz*S)
    m.translate([(-sx/2)*S, (-sy/2)*S, (-sz/2)*S])
    R = m.get_rotation_matrix_from_xyz([-slope_rad, 0, 0])
    m.rotate(R, center=(0,0,0))
    m.translate([cx*S, cy*S, cz*S])
    m.paint_uniform_color(color)
    m.compute_vertex_normals()
    return m

def merge(meshes):
    combined = meshes[0]
    for m in meshes[1:]:
        combined += m
    return combined

print('✅ ユーティリティ関数定義完了')

In [ ]:
# ============================================================
# CELL 3: カラーパレット
# ============================================================
C = {
    'pipe'    : [0.70, 0.72, 0.72],  # 単管（亜鉛メッキシルバー）
    'purlin'  : [0.75, 0.76, 0.72],  # 横垂木（φ31.8mm）
    'polica'  : [0.70, 0.90, 0.80],  # ポリカ波板（半透明グリーン）
    'noji'    : [0.78, 0.58, 0.30],  # 野地板
    'column'  : [0.52, 0.32, 0.14],  # 木柱（スギ）
    'keta'    : [0.58, 0.36, 0.16],  # 軒桁
    'taruki'  : [0.62, 0.40, 0.18],  # 垂木
    'wall_w'  : [0.72, 0.60, 0.40],  # 木造外壁（杉板）
    'wall_t'  : [0.85, 0.90, 0.80],  # 単管外壁（ポリカ）
    'dodai'   : [0.28, 0.16, 0.06],  # 土台（防腐材）
    'concrete': [0.72, 0.70, 0.68],  # 束石
    'ground'  : [0.18, 0.14, 0.09],  # 地面
    'glass'   : [0.75, 0.88, 0.94],  # 引き戸ガラス
    'sash'    : [0.70, 0.72, 0.72],  # アルミサッシ
}
print('✅ カラーパレット定義完了')

In [ ]:
# ============================================================
# CELL 4: 単管パイプ小屋ビルダー
# ============================================================
def build_tankan_shed(W=6000, D=5000, Hs=3000, Hn=2500,
                      col_pitch_x=1500, nokide=300,
                      label='単管パイプ小屋（本番 30㎡）'):
    """
    単管パイプ農作業小屋を生成
    W, D    : 幅・奥行(mm)
    Hs, Hn  : 南側・北側軒高(mm)
    """
    meshes = []
    slope  = np.arctan2(Hs - Hn, D)

    def col_h(y): return Hs + (Hn - Hs) * y / D
    def roof_z(y): return Hs + (Hn - Hs) * y / D

    x_list = list(range(0, W + 1, col_pitch_x))
    y_list = [0, D//2, D]

    # --- 地面 ---
    meshes.append(box(W/2, D/2, -5, W+2000, D+2000, 10, C['ground']))

    # --- スパイラル杭 ---
    for y in y_list:
        for x in x_list:
            meshes.append(cylinder(x, y, -450, 24.3, 900, C['pipe']))

    # --- 根太パイプ ---
    for y in y_list:
        meshes.append(cylinder(W/2, y, 0, 24.3, W+100, C['pipe'], axis='x'))
    for x in [0, W]:
        meshes.append(cylinder(x, D/2, 0, 24.3, D+100, C['pipe'], axis='y'))

    # --- 柱 ---
    for x in x_list:
        h = col_h(0); meshes.append(cylinder(x, 0, h/2, 24.3, h, C['pipe']))
        h = col_h(D); meshes.append(cylinder(x, D, h/2, 24.3, h, C['pipe']))
    for x in [0, W]:  # 妻面中間柱
        h = col_h(D/2); meshes.append(cylinder(x, D/2, h/2, 24.3, h, C['pipe']))

    # --- 軒桁（南・北）---
    meshes.append(cylinder(W/2, 0,   Hs, 24.3, W+nokide*2, C['pipe'], axis='x'))
    meshes.append(cylinder(W/2, D,   Hn, 24.3, W+nokide*2, C['pipe'], axis='x'))
    # 妻面桁（傾斜）
    for x in [0, W]:
        meshes.append(pipe_between([x,0,Hs], [x,D,Hn], 24.3, C['pipe']))

    # --- 横胴縁 ---
    for z_h in [800, 1600] if D <= 3000 else [800, 1600, 2400]:
        meshes.append(cylinder(W/2, 0, z_h, 24.3, W, C['pipe'], axis='x'))
        meshes.append(cylinder(W/2, D, z_h, 24.3, W, C['pipe'], axis='x'))

    # --- 筋交い ---
    meshes.append(pipe_between([50,0,50], [x_list[1]-50,0,Hs-50], 24.3, C['pipe']))
    meshes.append(pipe_between([50,D,50], [x_list[1]-50,D,Hn-50], 24.3, C['pipe']))

    # --- 垂木（φ48.6mm）---
    for x in x_list:
        ry = D + nokide * 2
        cz_r = (roof_z(-nokide) + roof_z(D+nokide)) / 2 + 26
        meshes.append(pipe_between(
            [x, -nokide, roof_z(-nokide)],
            [x, D+nokide, roof_z(D+nokide)], 24.3, C['pipe']))

    # --- 横垂木（φ31.8mm）@850mm ---
    n_pur = max(5, int((D + nokide*2) / 850) + 1)
    for i in range(n_pur):
        y  = -nokide + i * (D + nokide*2) / (n_pur-1)
        z  = roof_z(max(0, min(D, y))) + 26
        meshes.append(cylinder(W/2, y, z, 15.9, W+nokide*2, C['purlin'], axis='x'))

    # --- 野地板 ---
    ry  = D + nokide*2
    cz_n = (roof_z(-nokide) + roof_z(D+nokide)) / 2 + 40
    meshes.append(inclined_box(W/2, 0, cz_n, W+nokide*2, ry, 12, slope, C['noji']))

    # --- ポリカ屋根 ---
    cz_p = cz_n + 18
    meshes.append(inclined_box(W/2, 0, cz_p, W+nokide*2, ry, 8, slope, C['polica']))

    # --- 壁パネル ---
    dw, dh = (W*0.5 if W<=2000 else 1800), (Hs*0.9)  # 引き戸幅・高さ
    dx0 = (W - dw) / 2
    # 南面（引き戸あり）
    meshes.append(box(dx0/2,     4, Hs/2,    dx0,   8, Hs,    C['wall_t']))
    meshes.append(box(W-dx0/2,   4, Hs/2,    dx0,   8, Hs,    C['wall_t']))
    meshes.append(box(W/2,       4, dh+(Hs-dh)/2, dw, 8, Hs-dh, C['wall_t']))
    # 北面
    meshes.append(box(W/2, D-4, Hn/2, W, 8, Hn, C['wall_t']))
    # 東西妻面
    meshes.append(box(4,   D/2, Hn/2, 8, D, Hn, C['wall_t']))
    meshes.append(box(W-4, D/2, Hn/2, 8, D, Hn, C['wall_t']))
    # 引き戸ガラス
    meshes.append(box(W/2-dw/4, 4, dh/2, dw/2, 12, dh, C['glass']))
    meshes.append(box(W/2+dw/4, 4, dh/2, dw/2, 12, dh, C['glass']))

    return merge(meshes), label

print('✅ 単管パイプ小屋ビルダー定義完了')

In [ ]:
# ============================================================
# CELL 5: 木造農作業小屋ビルダー
# ============================================================
def build_wood_shed(W=6000, D=5000, Hs=3000, Hn=2500,
                    col_pitch_x=1500, nokide=300,
                    label='木造農作業小屋（本番 30㎡）'):
    """
    木造軸組農作業小屋を生成
    柱:105×105mm / 桁:105×150mm / 垂木:45×105mm / 束石:200×200×150mm
    """
    meshes = []
    dodai_top = 255  # 束石150 + 土台105
    slope     = np.arctan2(Hs - Hn, D)

    def col_h(y): return Hs + (Hn - Hs) * y / D
    def keta_z(y): return dodai_top + col_h(y) + 75  # 桁中心Z

    keta_S = keta_z(0)
    keta_N = keta_z(D)

    def roof_z(y): return keta_S + (keta_N - keta_S) * y / D + 75

    x_list = list(range(0, W + 1, col_pitch_x))
    y_list = [0, D//2, D]

    # --- 地面 ---
    meshes.append(box(W/2, D/2, -5, W+2000, D+2000, 10, C['ground']))

    # --- 束石 200×200×150mm ---
    for y in y_list:
        for x in x_list:
            meshes.append(box(x, y, 75, 200, 200, 150, C['concrete']))

    # --- 土台 105×105mm ---
    meshes.append(box(W/2, 0,   dodai_top-52.5, W+nokide*2, 105, 105, C['dodai']))
    meshes.append(box(W/2, D,   dodai_top-52.5, W+nokide*2, 105, 105, C['dodai']))
    meshes.append(box(0,   D/2, dodai_top-52.5, 105, D+nokide*2, 105, C['dodai']))
    meshes.append(box(W,   D/2, dodai_top-52.5, 105, D+nokide*2, 105, C['dodai']))
    meshes.append(box(W/2, D/2, dodai_top-52.5, W,  105, 105, C['dodai']))  # 中間

    # --- 柱 105×105mm ---
    for x in x_list:
        h = col_h(0); meshes.append(box(x, 0, dodai_top+h/2, 105, 105, h, C['column']))
        h = col_h(D); meshes.append(box(x, D, dodai_top+h/2, 105, 105, h, C['column']))
    for x in [0, W]:  # 妻面中間柱
        h = col_h(D/2); meshes.append(box(x, D/2, dodai_top+h/2, 105, 105, h, C['column']))

    # --- 軒桁 105×150mm ---
    meshes.append(box(W/2, 0, keta_S, W+nokide*2, 105, 150, C['keta']))
    meshes.append(box(W/2, D, keta_N, W+nokide*2, 105, 150, C['keta']))
    meshes.append(pipe_between([0,-nokide,keta_S], [0,D+nokide,keta_N], 52.5, C['keta']))
    meshes.append(pipe_between([W,-nokide,keta_S], [W,D+nokide,keta_N], 52.5, C['keta']))

    # --- 横胴縁 ---
    for z_h in [800, 1600]:
        meshes.append(box(W/2, 0, dodai_top+z_h, W, 105, 105, C['column']))
        meshes.append(box(W/2, D, dodai_top+z_h, W, 105, 105, C['column']))

    # --- 垂木 45×105mm ---
    for x in x_list:
        meshes.append(pipe_between(
            [x, -nokide,   roof_z(-nokide)],
            [x, D+nokide,  roof_z(D+nokide)],
            22.5, C['taruki']))

    # --- 野地板 ---
    ry   = D + nokide*2
    cz_n = (roof_z(-nokide) + roof_z(D+nokide)) / 2 + 16
    meshes.append(inclined_box(W/2, 0, cz_n, W+nokide*2, ry, 12, slope, C['noji']))

    # --- ポリカ屋根 ---
    cz_p = cz_n + 18
    meshes.append(inclined_box(W/2, 0, cz_p, W+nokide*2, ry, 8, slope, C['polica']))

    # --- 外壁（杉板）---
    dw, dh = (W*0.5 if W<=2000 else 1800), (Hs*0.9)
    dx0 = (W - dw) / 2
    # 南面
    meshes.append(box(dx0/2,     52.5, dodai_top+Hs/2, dx0, 105, Hs, C['wall_w']))
    meshes.append(box(W-dx0/2,   52.5, dodai_top+Hs/2, dx0, 105, Hs, C['wall_w']))
    meshes.append(box(W/2, 52.5, dodai_top+dh+(Hs-dh)/2, dw, 105, Hs-dh, C['wall_w']))
    # 北面
    meshes.append(box(W/2, D-52.5, dodai_top+Hn/2, W, 105, Hn, C['wall_w']))
    # 東西妻面
    meshes.append(box(52.5, D/2, dodai_top+Hn/2, 105, D, Hn, C['wall_w']))
    meshes.append(box(W-52.5, D/2, dodai_top+Hn/2, 105, D, Hn, C['wall_w']))
    # 引き戸ガラス
    meshes.append(box(W/2-dw/4, 52.5, dodai_top+dh/2, dw/2, 115, dh, C['glass']))
    meshes.append(box(W/2+dw/4, 52.5, dodai_top+dh/2, dw/2, 115, dh, C['glass']))

    return merge(meshes), label

print('✅ 木造農作業小屋ビルダー定義完了')

In [ ]:
# ============================================================
# CELL 6: 全4モデルを生成
# ============================================================
models = {
    'tankan_30sqm' : build_tankan_shed(
        W=6000, D=5000, Hs=3000, Hn=2500, col_pitch_x=1500,
        label='単管パイプ小屋（本番 30㎡ / 6000×5000mm）'),

    'tankan_mini'  : build_tankan_shed(
        W=1820, D=2730, Hs=2100, Hn=1800, col_pitch_x=910, nokide=200,
        label='単管パイプ小屋（練習版 約3畳 / 1820×2730mm）'),

    'wood_30sqm'   : build_wood_shed(
        W=6000, D=5000, Hs=3000, Hn=2500, col_pitch_x=1500,
        label='木造農作業小屋（本番 30㎡ / 6000×5000mm）'),

    'wood_mini'    : build_wood_shed(
        W=1820, D=2730, Hs=2100, Hn=1800, col_pitch_x=910, nokide=200,
        label='木造農作業小屋（練習版 約3畳 / 1820×2730mm）'),
}

for key, (mesh, label) in models.items():
    verts = len(mesh.vertices)
    tris  = len(mesh.triangles)
    print(f'  ✅ {label}')
    print(f'     頂点: {verts:,}  三角形: {tris:,}')

In [ ]:
# ============================================================
# CELL 7: レンダリング関数（Colab・ローカル共通）
# ============================================================
def render_views(mesh, label, filename=None):
    """4アングル（パース・正面・側面・平面）をmatplotlibで描画"""
    bounds  = mesh.get_axis_aligned_bounding_box()
    center  = bounds.get_center()
    extent  = bounds.get_max_bound() - bounds.get_min_bound()
    dist    = float(max(extent)) * 1.8

    cam_configs = [
        # (eye_offset,         up,         subplot_title)
        ([ 0.7, -0.7,  0.5],  [0, 0, 1],  '完成パース（南西45°）'),
        ([ 0.0, -1.0,  0.0],  [0, 0, 1],  '正面図（南面）'),
        ([-1.0,  0.0,  0.0],  [0, 0, 1],  '側面図（西面）'),
        ([ 0.0,  0.0,  1.0],  [0, 1, 0],  '平面図（真上）'),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.patch.set_facecolor('#0d1117')
    fig.suptitle(label, color='white', fontsize=14,
                 fontproperties='Noto Sans CJK JP' if 'CJK' in plt.rcParams.get('font.family', [''])[0] else None,
                 y=0.98)

    renderer = o3d.visualization.rendering.OffscreenRenderer(900, 600)
    mat = o3d.visualization.rendering.MaterialRecord()
    mat.shader = 'defaultLit'

    for ax, (eye_off, up, title) in zip(axes.flat, cam_configs):
        renderer.scene.clear_geometry()
        renderer.scene.add_geometry('shed', mesh, mat)
        renderer.scene.set_background([0.08, 0.10, 0.14, 1.0])
        renderer.scene.scene.enable_sun_light(True)
        renderer.scene.scene.set_sun_light(
            [0.5, -0.6, -0.8], [1.4, 1.3, 1.2], 80000)

        eye = np.array(eye_off) * dist + center
        renderer.setup_camera(55.0, center, eye, np.array(up, float))

        img = np.asarray(renderer.render_to_image())
        ax.imshow(img)
        ax.set_title(title, color='#a0b4c8', fontsize=11, pad=6)
        ax.axis('off')
        ax.set_facecolor('#0d1117')

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    if filename:
        plt.savefig(filename, dpi=120, bbox_inches='tight',
                    facecolor='#0d1117')
        print(f'  💾 保存: {filename}')
    plt.show()

print('✅ レンダリング関数定義完了')

In [ ]:
# ============================================================
# CELL 8: 4アングルレンダリング — 全4モデル
# ============================================================
for key, (mesh, label) in models.items():
    print(f'\n🔨 レンダリング中: {label}')
    render_views(mesh, label, filename=f'render_{key}.png')

print('\n✅ 全4モデルのレンダリング完了！')

In [ ]:
# ============================================================
# CELL 9: .ply / .obj 形式で保存（Meshlab等で開ける）
# ============================================================
for key, (mesh, label) in models.items():
    ply_path = f'{key}.ply'
    o3d.io.write_triangle_mesh(ply_path, mesh)
    size_kb = os.path.getsize(ply_path) // 1024
    print(f'✅ {ply_path}  ({size_kb} KB)')

print('\n📦 .ply ファイルはMeshlab・Blender・Open3Dで読み込めます')

In [ ]:
# ============================================================
# CELL 10: ローカル専用 — インタラクティブビューア
# ============================================================
# Colabでは動きません。ローカルのJupyterで実行してください。
#
#   pip install open3d
#   jupyter notebook view_models_open3d.ipynb

def draw_shed_interactive(key='tankan_mini'):
    """指定モデルをインタラクティブウィンドウで表示"""
    mesh, label = models[key]
    print(f'🏡 {label}')
    print('  左ドラッグ : 回転')
    print('  右ドラッグ : 平行移動')
    print('  スクロール : ズーム')
    print('  Ctrl+C / Q : 終了')
    o3d.visualization.draw_geometries(
        [mesh],
        window_name=f'ナナカファーム — {label}',
        width=1280, height=800, left=80, top=40
    )

if IS_COLAB:
    print('⚠️  Colab環境: インタラクティブウィンドウは開けません')
    print('   ローカルで実行するには:')
    print('   draw_shed_interactive("tankan_mini")   # 単管小屋 約3畳')
    print('   draw_shed_interactive("tankan_30sqm")  # 単管小屋 30㎡')
    print('   draw_shed_interactive("wood_mini")     # 木造小屋 約3畳')
    print('   draw_shed_interactive("wood_30sqm")    # 木造小屋 30㎡')
else:
    # ローカル実行時は自動で単管縮小版を開く
    draw_shed_interactive('tankan_mini')

## 📐 設計仕様メモ

| 項目 | 単管（本番 30㎡） | 単管（約3畳） | 木造（本番 30㎡） | 木造（約3畳） |
|------|----------------|--------------|-----------------|-------------|
| 幅×奥行 | 6000×5000mm | 1820×2730mm | 6000×5000mm | 1820×2730mm |
| 南側軒高 | 3,000mm | 2,100mm | 3,000mm | 2,100mm |
| 北側軒高 | 2,500mm | 1,800mm | 2,500mm | 1,800mm |
| 屋根勾配 | 1/10（5.7°） | 1/9（6.3°） | 1/10（5.7°） | 1/9（6.3°） |
| 主材料 | φ48.6mm 単管 | φ48.6mm 単管 | 105×105mm 木柱 | 105×105mm 木柱 |
| 基礎 | スパイラル杭 | スパイラル杭 | 束石 200×200×150 | 束石 200×200×150 |
| 概算費用 | ¥237,100〜 | ¥79,000〜 | — | ¥113,000〜 |

---

**参考資料**
- Zenn記事: https://zenn.dev/hiroakikody/articles/47ba166f4dad26
- 農研機構「建設足場資材利用園芸ハウス施工マニュアル」(2017)
- GitHubリポジトリ: https://github.com/hiroakikody/nanaka-farm-shed-designs

*七花ファーム（ナナカファーム）熊本県 / 古閑 弘晃*